# Week 1 Day 5 — Final Model Validation, Deployment Readiness & Project Completion
## Adult Income Classification

**Primary metric:** F1-score | **Random state:** 42 | **Split:** 60% train / 20% development / 20% untouched final test

This notebook covers all Day 5 tasks:
1. Final Model Validation
2. Error Analysis
3. Model Interpretation
4. Production Inference

The final test set is **never** used for model selection or threshold selection — it is evaluated exactly once, at the end, after the model and decision threshold have already been locked in on the development set.

## Setup

In [ ]:
import os, sys, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

RESULTS = Path('results')
FIGURES = Path('figures')
MODELS = Path('models')
for p in [RESULTS, FIGURES, MODELS]:
    p.mkdir(exist_ok=True)

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__, '| numpy:', np.__version__,
      '| scikit-learn:', sklearn.__version__, '| joblib:', joblib.__version__)

## Task 1 — Final Model Validation

**Objective:** predict whether annual income is `>50K` (1) or `<=50K` (0).

### Data loading

In [ ]:
COLUMNS = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
           'marital-status', 'occupation', 'relationship', 'race', 'sex',
           'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


def load_adult():
    """Load the Adult Income dataset from a local file if available,
    otherwise fall back to fetching it from OpenML."""
    candidates = ['adult.data', 'adult.csv', 'adult_income.csv', 'adult_income_data.csv']
    d = None
    source = None

    for path in candidates:
        if not os.path.exists(path):
            continue
        if path.endswith('.data'):
            d = pd.read_csv(path, header=None, names=COLUMNS,
                             skipinitialspace=True, na_values='?')
        else:
            raw = pd.read_csv(path, skipinitialspace=True, na_values='?')
            if set(COLUMNS).issubset(raw.columns):
                d = raw[COLUMNS].copy()
            elif raw.shape[1] == 15:
                raw.columns = COLUMNS
                d = raw.copy()
            else:
                raise ValueError(f'Unexpected columns in {path}')
        source = path
        break

    if d is None:
        from sklearn.datasets import fetch_openml
        d = fetch_openml('adult', version=2, as_frame=True).frame.copy()
        source = 'OpenML Adult v2'
        target = 'class' if 'class' in d.columns else 'income'
        d = d.rename(columns={
            target: 'income',
            'education_num': 'education-num',
            'marital_status': 'marital-status',
            'capital_gain': 'capital-gain',
            'capital_loss': 'capital-loss',
            'hours_per_week': 'hours-per-week',
            'native_country': 'native-country',
        })
        d = d[COLUMNS]

    for c in d.select_dtypes(include='object').columns:
        d[c] = d[c].astype(str).str.strip().replace({'?': np.nan, 'nan': np.nan})

    d['income'] = d['income'].astype(str).str.strip()
    d['target'] = d['income'].map({'<=50K': 0, '>50K': 1, '<=50K.': 0, '>50K.': 1})
    dropped = int(d['target'].isna().sum())
    d = d.dropna(subset=['target']).copy()
    d['target'] = d['target'].astype(int)
    d = d.drop(columns=['income'])
    return d, source, dropped


df, data_source, dropped = load_adult()
print('Data source:', data_source, '| shape:', df.shape, '| dropped target rows:', dropped)

In [ ]:
numeric_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = ['workclass', 'education', 'marital-status', 'occupation',
                         'relationship', 'race', 'sex', 'native-country']

X = df.drop(columns='target')
y = df['target'].astype(int)
assert set(X.columns) == set(numeric_features + categorical_features)

print('Target distribution:')
target_table = y.value_counts().sort_index().rename_axis('target').reset_index(name='count')
target_table['proportion'] = (target_table['count'] / len(y)).round(4)
display(target_table)

missing_table = pd.DataFrame({
    'Missing_Count': X.isna().sum(),
    'Missing_Percent': (X.isna().mean() * 100).round(2),
}).sort_values('Missing_Count', ascending=False)
display(missing_table)

target_table.to_csv(RESULTS / 'target_distribution.csv', index=False)
missing_table.to_csv(RESULTS / 'missing_values.csv')

### 60 / 20 / 20 leakage-safe split

In [ ]:
X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev, y_train_dev, test_size=0.25, stratify=y_train_dev, random_state=RANDOM_STATE)

train_ids, dev_ids, test_ids = set(X_train.index), set(X_dev.index), set(X_test.index)
assert train_ids.isdisjoint(dev_ids) and train_ids.isdisjoint(test_ids) and dev_ids.isdisjoint(test_ids)

print('Split sizes -> train:', len(X_train), '| dev:', len(X_dev), '| test:', len(X_test))
print('Leakage check: PASS — train/dev/test indices are disjoint.')

### Preprocessing — kept entirely inside the pipeline

In [ ]:
try:
    enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    enc = OneHotEncoder(handle_unknown='ignore', sparse=False)

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', enc),
])
preprocessor = ColumnTransformer([
    ('numeric', num_pipe, numeric_features),
    ('categorical', cat_pipe, categorical_features),
])

### Load the Day 4 artifact (required Day 5 step), with a reproducible fallback

In [ ]:
artifact_candidates = [
    Path('adult_income_final_pipeline.joblib'),
    MODELS / 'adult_income_final_pipeline.joblib',
    Path('final_model.joblib'),
    MODELS / 'final_model.joblib',
]
artifact_path = next((p for p in artifact_candidates if p.exists()), None)

day4_loaded = False
final_model = None
selected_threshold = 0.50
final_model_name = None
saved_parameters = {}

if artifact_path:
    artifact = joblib.load(artifact_path)
    day4_loaded = True
    if isinstance(artifact, dict):
        final_model = artifact.get('model')
        selected_threshold = float(artifact.get('threshold', 0.50))
        final_model_name = artifact.get('model_name', 'Saved Final Model')
        saved_parameters = artifact.get('best_parameters', {})
    else:
        final_model = artifact
        selected_threshold = 0.50
        final_model_name = 'Saved Pipeline'

    if final_model is None:
        raise ValueError('Day 4 artifact dictionary has no "model" key.')

    print('Day 4 artifact loaded:', artifact_path)
    print('Model:', final_model_name, '| threshold:', selected_threshold)
else:
    print('Day 4 artifact not found — running reproducible fallback tuning.')

### Fallback tuning (only runs if no Day 4 artifact was found; uses training data only)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    'Logistic Regression': Pipeline([('preprocessor', preprocessor),
                                      ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
    'Random Forest': Pipeline([('preprocessor', preprocessor),
                                ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))]),
    'Gradient Boosting': Pipeline([('preprocessor', preprocessor),
                                    ('model', GradientBoostingClassifier(random_state=RANDOM_STATE))]),
}

params = {
    'Logistic Regression': {
        'model__C': np.logspace(-3, 2, 10),
        'model__class_weight': [None, 'balanced'],
        'model__solver': ['lbfgs'],
    },
    'Random Forest': {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 10, 20, 30],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': ['sqrt', 'log2', None],
    },
    'Gradient Boosting': {
        'model__n_estimators': [50, 100, 150, 200],
        'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
        'model__max_depth': [2, 3, 4, 5],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
    },
}

best_models = {}
tuning_rows = []
development_comparison = None

if not day4_loaded:
    for name, est in models.items():
        print('Tuning', name)
        search = RandomizedSearchCV(est, params[name], n_iter=20, scoring='f1', cv=cv,
                                     random_state=RANDOM_STATE, n_jobs=-1, return_train_score=True)
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        tuning_rows.append({'Model': name, 'Best_CV_F1': search.best_score_,
                             'Best_Parameters': str(search.best_params_)})
        print('Best CV F1:', round(search.best_score_, 4), '| params:', search.best_params_)

    tuning_summary = pd.DataFrame(tuning_rows).sort_values('Best_CV_F1', ascending=False)
    tuning_summary.to_csv(RESULTS / 'hyperparameter_search_summary.csv', index=False)
    display(tuning_summary)

    dev_rows = []
    for name, m in best_models.items():
        p = m.predict_proba(X_dev)[:, 1]
        pred = (p >= 0.50).astype(int)
        dev_rows.append({
            'Model': name, 'Threshold': 0.50,
            'Accuracy': accuracy_score(y_dev, pred),
            'Precision': precision_score(y_dev, pred, zero_division=0),
            'Recall': recall_score(y_dev, pred, zero_division=0),
            'F1': f1_score(y_dev, pred, zero_division=0),
            'ROC-AUC': roc_auc_score(y_dev, p),
            'PR-AUC': average_precision_score(y_dev, p),
            'Brier': brier_score_loss(y_dev, p),
        })
    development_comparison = pd.DataFrame(dev_rows).sort_values('F1', ascending=False)
    development_comparison.to_csv(RESULTS / 'development_model_comparison.csv', index=False)
    display(development_comparison)

    final_model_name = development_comparison.iloc[0]['Model']
    final_model = best_models[final_model_name]
else:
    # Preserve the prior Day 4 comparison table if it exists, otherwise create a transparent placeholder.
    if (RESULTS / 'development_model_comparison.csv').exists():
        development_comparison = pd.read_csv(RESULTS / 'development_model_comparison.csv')
    else:
        development_comparison = pd.DataFrame([{
            'Model': final_model_name, 'Threshold': selected_threshold,
            'Accuracy': np.nan, 'Precision': np.nan, 'Recall': np.nan, 'F1': np.nan,
            'ROC-AUC': np.nan, 'PR-AUC': np.nan, 'Brier': np.nan,
        }])
        development_comparison.to_csv(RESULTS / 'development_model_comparison.csv', index=False)
    display(development_comparison)

### Decision threshold — selected on the development set only

In [ ]:
dev_probabilities = final_model.predict_proba(X_dev)[:, 1]

if not day4_loaded:
    rows = []
    for t in np.arange(0.10, 0.91, 0.01):
        pred = (dev_probabilities >= t).astype(int)
        rows.append({
            'Threshold': round(float(t), 2),
            'Accuracy': accuracy_score(y_dev, pred),
            'Precision': precision_score(y_dev, pred, zero_division=0),
            'Recall': recall_score(y_dev, pred, zero_division=0),
            'F1': f1_score(y_dev, pred, zero_division=0),
        })
    threshold_df = pd.DataFrame(rows)
    best_row = threshold_df.sort_values(['F1', 'Precision', 'Recall', 'Threshold'],
                                         ascending=[False, False, False, True]).iloc[0]
    selected_threshold = float(best_row['Threshold'])
    threshold_df.to_csv(RESULTS / 'threshold_analysis.csv', index=False)
    display(threshold_df)

    plt.figure(figsize=(8, 5))
    plt.plot(threshold_df.Threshold, threshold_df.Precision, label='Precision')
    plt.plot(threshold_df.Threshold, threshold_df.Recall, label='Recall')
    plt.plot(threshold_df.Threshold, threshold_df.F1, label='F1')
    plt.axvline(selected_threshold, ls='--', label=f'Selected={selected_threshold:.2f}')
    plt.xlabel('Threshold')
    plt.ylabel('Score')
    plt.title('Development Threshold Analysis')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(FIGURES / 'threshold_analysis.png', dpi=150)
    plt.show()
else:
    print('Using the stored Day 4 threshold —', selected_threshold, '(not retuned on test data).')

### Pipeline verification before touching the final test set

In [ ]:
probe = X_test.head(5)
probe_p = final_model.predict_proba(probe)[:, 1]
probe_y = (probe_p >= selected_threshold).astype(int)
assert len(probe_p) == 5 and np.all((probe_p >= 0) & (probe_p <= 1))
print('Complete pipeline verification: PASS')

### Final test set — single locked evaluation

In [ ]:
test_probabilities = final_model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= selected_threshold).astype(int)

final_metrics = {
    'Model': final_model_name, 'Threshold': selected_threshold,
    'Accuracy': accuracy_score(y_test, test_predictions),
    'Precision': precision_score(y_test, test_predictions, zero_division=0),
    'Recall': recall_score(y_test, test_predictions, zero_division=0),
    'F1': f1_score(y_test, test_predictions, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, test_probabilities),
    'PR-AUC': average_precision_score(y_test, test_probabilities),
    'Brier': brier_score_loss(y_test, test_probabilities),
}
final_test_df = pd.DataFrame([final_metrics])
final_test_df.to_csv(RESULTS / 'final_test_metrics.csv', index=False)

print('FINAL TEST RESULTS')
display(final_test_df)
print(classification_report(y_test, test_predictions, target_names=['<=50K', '>50K'], zero_division=0))

## Task 2 — Error Analysis

In [ ]:
cm = confusion_matrix(y_test, test_predictions)
tn, fp, fn, tp = cm.ravel()
pd.DataFrame([{'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}]).to_csv(RESULTS / 'confusion_matrix_summary.csv', index=False)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title('Confusion Matrix — Final Test Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks([0, 1], ['<=50K', '>50K'])
plt.yticks([0, 1], ['<=50K', '>50K'])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.tight_layout()
plt.savefig(FIGURES / 'confusion_matrix.png', dpi=150)
plt.show()
print('TN:', tn, '| FP:', fp, '| FN:', fn, '| TP:', tp)

In [ ]:
errors = X_test.copy()
errors['actual'] = y_test.values
errors['predicted'] = test_predictions
errors['probability'] = test_probabilities
errors['error_type'] = np.select(
    [(errors.actual == 0) & (errors.predicted == 1),
     (errors.actual == 1) & (errors.predicted == 0)],
    ['False Positive', 'False Negative'],
    default='Correct',
)

fp_df = errors[errors.error_type == 'False Positive']
fn_df = errors[errors.error_type == 'False Negative']

errors.to_csv(RESULTS / 'test_error_analysis.csv', index=False)
fp_df.head(20).to_csv(RESULTS / 'false_positive_samples.csv', index=False)
fn_df.head(20).to_csv(RESULTS / 'false_negative_samples.csv', index=False)

print('False positives:', len(fp_df), '| False negatives:', len(fn_df))
display(fp_df.head(10))
display(fn_df.head(10))

### Subgroup analysis

In [ ]:
def subgroup_analysis(Xg, yg, p, col, threshold, min_n=30):
    """Break down accuracy/F1 and predicted-positive rate by a categorical column,
    skipping groups smaller than min_n so the numbers stay meaningful."""
    z = Xg[[col]].copy()
    z['actual'] = np.asarray(yg)
    z['pred'] = (p >= threshold).astype(int)

    rows = []
    for value, g in z.groupby(col, dropna=False):
        if len(g) < min_n:
            continue
        rows.append({
            'Group': value,
            'N': len(g),
            'Actual_Positive_Rate': g['actual'].mean(),
            'Predicted_Positive_Rate': g['pred'].mean(),
            'Accuracy': accuracy_score(g['actual'], g['pred']),
            'F1': f1_score(g['actual'], g['pred'], zero_division=0),
        })
    return pd.DataFrame(rows).sort_values('N', ascending=False)


for col in ['sex', 'race', 'education']:
    print(f'--- Subgroup analysis: {col} ---')
    sub_df = subgroup_analysis(X_test, y_test, test_probabilities, col, selected_threshold)
    sub_df.to_csv(RESULTS / f'subgroup_analysis_{col}.csv', index=False)
    display(sub_df)

## Task 3 — Model Interpretation

Learning curve (bias/variance diagnostic), calibration curve, ROC / Precision-Recall curves,
and feature importance / coefficients — all computed without touching the final test set
except for the ROC/PR curves, which describe the already-locked final evaluation.

In [ ]:
train_sizes, train_scores, cv_scores = learning_curve(
    final_model, X_train, y_train, cv=cv if not day4_loaded else 5,
    scoring='f1', random_state=RANDOM_STATE,
    train_sizes=np.linspace(0.1, 1.0, 6), n_jobs=-1,
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Training F1')
plt.plot(train_sizes, cv_scores.mean(axis=1), 'o-', label='Cross-val F1')
plt.xlabel('Training examples')
plt.ylabel('F1-score')
plt.title('Learning Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURES / 'learning_curve.png', dpi=150)
plt.show()

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, test_probabilities, n_bins=10)

plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, 'o-', label=final_model_name)
plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfectly calibrated')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.title('Calibration Curve — Final Test Set')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURES / 'calibration_curve.png', dpi=150)
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_probabilities)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_probabilities)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, label=f'ROC-AUC = {final_metrics["ROC-AUC"]:.3f}')
axes[0].plot([0, 1], [0, 1], '--', color='gray')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(recall_curve, precision_curve, label=f'PR-AUC = {final_metrics["PR-AUC"]:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(FIGURES / 'roc_pr_curves.png', dpi=150)
plt.show()

In [ ]:
def get_feature_names(model):
    """Best-effort retrieval of post-preprocessing feature names from a fitted pipeline."""
    try:
        return list(model.named_steps['preprocessor'].get_feature_names_out())
    except Exception:
        return None


feature_names = get_feature_names(final_model)
estimator = final_model.named_steps.get('model', final_model) if hasattr(final_model, 'named_steps') else final_model

importance_df = None
if feature_names is not None:
    if hasattr(estimator, 'feature_importances_'):
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': estimator.feature_importances_,
        }).sort_values('Importance', ascending=False)
    elif hasattr(estimator, 'coef_'):
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': estimator.coef_.ravel(),
        }).sort_values('Importance', key=np.abs, ascending=False)

if importance_df is not None:
    importance_df.to_csv(RESULTS / 'feature_importance.csv', index=False)
    top20 = importance_df.head(20)

    plt.figure(figsize=(8, 8))
    plt.barh(top20['Feature'][::-1], top20['Importance'][::-1])
    plt.xlabel('Importance')
    plt.title(f'Top 20 Feature Importances — {final_model_name}')
    plt.tight_layout()
    plt.savefig(FIGURES / 'feature_importance.png', dpi=150)
    plt.show()
    display(top20)
else:
    print('Feature importance not available for this model/pipeline structure.')

## Task 4 — Production Inference

In [ ]:
def predict_income(new_data, model=None, threshold=None):
    """Run the trained pipeline on new, raw examples with the same 14 input columns."""
    m = model if model is not None else final_model
    t = threshold if threshold is not None else selected_threshold
    p = m.predict_proba(new_data)[:, 1]
    pred = (p >= float(t)).astype(int)
    return pd.DataFrame({
        'probability_gt_50K': p,
        'prediction': pred,
        'predicted_income': np.where(pred == 1, '>50K', '<=50K'),
    })


# 10 unseen/raw examples
examples = pd.DataFrame([
    {'age': 39, 'workclass': 'State-gov', 'fnlwgt': 77516, 'education': 'Bachelors', 'education-num': 13,
     'marital-status': 'Never-married', 'occupation': 'Adm-clerical', 'relationship': 'Not-in-family',
     'race': 'White', 'sex': 'Male', 'capital-gain': 2174, 'capital-loss': 0, 'hours-per-week': 40,
     'native-country': 'United-States'},
    {'age': 50, 'workclass': 'Private', 'fnlwgt': 83311, 'education': 'Masters', 'education-num': 14,
     'marital-status': 'Married-civ-spouse', 'occupation': 'Exec-managerial', 'relationship': 'Husband',
     'race': 'White', 'sex': 'Male', 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 50,
     'native-country': 'United-States'},
    {'age': 28, 'workclass': 'Private', 'fnlwgt': 123456, 'education': 'Bachelors', 'education-num': 13,
     'marital-status': 'Never-married', 'occupation': 'Tech-support', 'relationship': 'Not-in-family',
     'race': 'Asian-Pac-Islander', 'sex': 'Female', 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 40,
     'native-country': 'United-States'},
    {'age': 45, 'workclass': 'Self-emp-not-inc', 'fnlwgt': 190000, 'education': 'HS-grad', 'education-num': 9,
     'marital-status': 'Married-civ-spouse', 'occupation': 'Sales', 'relationship': 'Husband',
     'race': 'White', 'sex': 'Male', 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 60,
     'native-country': 'United-States'},
    {'age': 23, 'workclass': 'Private', 'fnlwgt': 150000, 'education': 'Some-college', 'education-num': 10,
     'marital-status': 'Never-married', 'occupation': 'Sales', 'relationship': 'Own-child',
     'race': 'White', 'sex': 'Female', 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 35,
     'native-country': 'United-States'},
    {'age': 60, 'workclass': 'Private', 'fnlwgt': 210000, 'education': 'Masters', 'education-num': 14,
     'marital-status': 'Married-civ-spouse', 'occupation': 'Prof-specialty', 'relationship': 'Husband',
     'race': 'White', 'sex': 'Male', 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 45,
     'native-country': 'United-States'},
    {'age': 34, 'workclass': 'Private', 'fnlwgt': 101010, 'education': 'Bachelors', 'education-num': 13,
     'marital-status': 'Divorced', 'occupation': 'Prof-specialty', 'relationship': 'Unmarried',
     'race': 'Black', 'sex': 'Female', 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 40,
     'native-country': 'United-States'},
    {'age': 52, 'workclass': 'Private', 'fnlwgt': 180000, 'education': 'Doctorate', 'education-num': 16,
     'marital-status': 'Married-civ-spouse', 'occupation': 'Prof-specialty', 'relationship': 'Husband',
     'race': 'White', 'sex': 'Male', 'capital-gain': 99999, 'capital-loss': 0, 'hours-per-week': 50,
     'native-country': 'United-States'},
    {'age': 31, 'workclass': 'Private', 'fnlwgt': 140000, 'education': 'HS-grad', 'education-num': 9,
     'marital-status': 'Never-married', 'occupation': 'Craft-repair', 'relationship': 'Not-in-family',
     'race': 'White', 'sex': 'Male', 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 40,
     'native-country': 'United-States'},
    {'age': 42, 'workclass': 'Private', 'fnlwgt': 160000, 'education': 'Bachelors', 'education-num': 13,
     'marital-status': 'Married-civ-spouse', 'occupation': 'Exec-managerial', 'relationship': 'Wife',
     'race': 'White', 'sex': 'Female', 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 45,
     'native-country': 'United-States'},
])

inference_results = predict_income(examples)
inference_results.to_csv(RESULTS / 'inference_examples.csv', index=False)
display(inference_results)
assert len(inference_results) == 10

### Save the deployment artifact, inference script, and requirements file

In [ ]:
artifact_out = {
    'model': final_model,
    'threshold': selected_threshold,
    'model_name': final_model_name,
    'best_parameters': saved_parameters,
}
joblib.dump(artifact_out, MODELS / 'adult_income_final_pipeline.joblib')

Path('inference.py').write_text('''import joblib
import numpy as np
import pandas as pd


def predict_income(new_data, artifact_path="models/adult_income_final_pipeline.joblib"):
    # Load the saved pipeline + threshold and score new, raw examples.
    artifact = joblib.load(artifact_path)
    p = artifact["model"].predict_proba(new_data)[:, 1]
    pred = (p >= float(artifact["threshold"])).astype(int)
    return pd.DataFrame({
        "probability_gt_50K": p,
        "prediction": pred,
        "predicted_income": np.where(pred == 1, ">50K", "<=50K"),
    })
''', encoding='utf-8')

Path('requirements.txt').write_text(
    f"numpy=={np.__version__}\n"
    f"pandas=={pd.__version__}\n"
    f"scikit-learn=={sklearn.__version__}\n"
    f"joblib=={joblib.__version__}\n"
    f"matplotlib=={__import__('matplotlib').__version__}\n",
    encoding='utf-8',
)

print('DAY 5 COMPLETE — saved artifact, requirements.txt, inference.py, CSV results and PNG figures.')

## Final Submission Checklist
- [x] `adult_income_final_pipeline.joblib`
- [x] Final Jupyter notebook
- [x] Final metrics table
- [x] Confusion matrix
- [x] Learning curve
- [x] Calibration curve
- [x] ROC and Precision–Recall curves
- [x] Feature interpretation visualization
- [x] False-positive / false-negative samples
- [x] Subgroup analysis
- [x] Working inference function + 10 unseen examples
- [x] `requirements.txt`
- [x] Leakage verification
- [x] Final test evaluation after model/threshold lock